# A Data-Centric Pipeline for Label Error Detection and Correction

**question:** Can systematic detection and correction of mislabeled training examples improve
sentiment classification while the model architecture and training configuration stay fixed?

**Design:** IMDb train, inject known label noise, Cleanlab (on frozen-BERT embeddings + out-of-fold
LogisticRegression), clean (remove / relabel), fine-tune the *same* `bert-base-uncased`, evaluate on the
*same, untouched* test set.

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/MUKAMAFrancois/data-centric-ai.git"
REPO_DIR = "data-centric-ai"

if not os.path.exists("src"):         
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")

### Persist results across disconnects
Free Colab sessions die. With `USE_DRIVE = True`, results and the embedding cache are written to Google Drive,
so re-running the notebook resumes where it stopped.

In [ ]:
from src.utils import load_config, set_seed

cfg = load_config("config.yaml")
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/data-centric-ai"
    cfg["paths"].update({
        "cache_dir": f"{base}/cache",
        "results_file": f"{base}/experiments.csv",
        "issues_dir": f"{base}/label_issues",
    })

set_seed(cfg["seed"])
cfg["paths"]

In [ ]:
# Pilot switch: True for a quick ~20-min check, False for the full experiment
PILOT = True

if PILOT:
    cfg["noise"]["rates"] = [0.2]
    cfg["noise"]["seeds"] = [42]
    cfg["cleaning"]["strategies"] = ["remove"]

print("noise:", cfg["noise"], "| strategies:", cfg["cleaning"]["strategies"])

## 2. Load data

In [ ]:
from src.data import load_imdb, profile_data, split_overlap

train_df, test_df = load_imdb(
    cfg["data"]["dataset_name"],
    cfg["data"]["train_subset"],
    cfg["data"]["test_subset"], 
    cfg["seed"])
print(f"train: {len(train_df):,}   test: {len(test_df):,}")
train_df.head()

## 3. Data profiling (baseline data quality)

In [ ]:
import pandas as pd

pd.DataFrame({"train": profile_data(train_df), "test": profile_data(test_df)})

In [ ]:
print("Exact train/test text overlap (leakage check):", split_overlap(train_df, test_df))

## 4. Controlled label noise
We flip the same fraction of labels in each class and keep the original label in `true_label`.
This known ground truth is what lets us grade Cleanlab objectively.

In [ ]:
from src.data import inject_noise

demo = inject_noise(train_df, 0.10, seed=42)
print(f"flipped: {demo.is_noisy.sum():,} of {len(demo):,}  |  class balance after: {demo.label.mean():.3f}")
demo[demo.is_noisy][["id", "true_label", "label", "is_noisy", "text"]].head()

## 5. Frozen BERT embeddings (for the detector)
Frozen = BERT never sees our labels, so it can't memorise the noise. Computed once (~5-10 min on a T4) and cached.

In [ ]:
from pathlib import Path
from src.model import embed_texts

det_cfg = cfg["detection"]
cache = Path(cfg["paths"]["cache_dir"]) / f"emb_{det_cfg['embed_model']}_{len(train_df)}_{det_cfg['embed_max_length']}.npy"
features = embed_texts(train_df["text"].tolist(), det_cfg["embed_model"], det_cfg["embed_max_length"],det_cfg["embed_batch_size"], str(cache))
features.shape

## 6. Label-error detection (Level 1: data quality)
Cheap: seconds per run. Out-of-fold LogisticRegression probabilities → Cleanlab → compare flags with the
labels we actually corrupted.

In [ ]:
from src.pipeline import detect

det_rows = []
for rate in cfg["noise"]["rates"]:
    for seed in cfg["noise"]["seeds"]:
        _, _, det = detect(train_df, features, rate, seed, det_cfg)
        det_rows.append({"noise_rate": rate, "seed": seed, **det})

det_table = pd.DataFrame(det_rows)
det_table.groupby("noise_rate")[["n_true_errors", "n_flagged", "det_precision", "det_recall", "det_f1",                            "review_reduction"]].mean().round(3)

### errors in the *original* IMDb labels
At 0% injected noise, anything Cleanlab flags is a candidate error in the real dataset. Read a few:
some will be genuinely mislabeled or ambiguous reviews.

In [ ]:
from src.cleaning import find_issues, get_oof_probs

labels0 = train_df["label"].to_numpy()
issues0 = find_issues(labels0, get_oof_probs(features, labels0, det_cfg["cv_folds"], cfg["seed"]))
suspects = pd.concat([train_df, issues0], axis=1).query("is_label_issue").sort_values("label_quality")
print(f"{len(suspects):,} suspicious labels in the original training subset")
pd.set_option("display.max_colwidth", 300)
suspects[["id", "label", "suggested_label", "label_quality", "text"]].head(10)

## 7. Train the SAME BERT on clean / noisy / cleaned data (Level 2: model quality)
Every run uses identical hyperparameters from `config.yaml`. Each finished run is appended to the results
CSV, so if Colab disconnects, just re-run the notebook: finished runs are skipped.

Rough cost on a T4 with 10k train reviews: ~6-10 min per run, 30 runs in total.

In [ ]:
from src.model import fine_tune_and_predict
from src.pipeline import run_experiments

results = run_experiments(cfg, train_df, test_df, features, fine_tune_and_predict,results_file=cfg["paths"]["results_file"],issues_dir=cfg["paths"]["issues_dir"])
results.tail()

## 8. Results

In [ ]:
from src.evaluate import recovery_table, summarize

results = pd.read_csv(cfg["paths"]["results_file"])
summarize(results, metrics=("accuracy", "precision", "recall", "f1"))

In [ ]:
recovery_table(results, "f1")

In [ ]:
from src.evaluate import plot_detection, plot_metric_vs_noise

fig_dir = cfg["paths"]["figures_dir"]
plot_metric_vs_noise(results, "f1", f"{fig_dir}/f1_vs_noise.png");
plot_detection(results, f"{fig_dir}/detection_vs_noise.png");

In [ ]:
from src.evaluate import plot_confusion

RATE, SEED = 0.2, cfg["noise"]["seeds"][0]
for name in ["noisy", "cleaned_relabel"]:
    row = results.query("noise_rate == @RATE and seed == @SEED and dataset == @name").iloc[0]
    plot_confusion(row, f"{name} @ {RATE:.0%} noise", f"{fig_dir}/cm_{name}_{int(RATE*100)}.png")

## 9. Error analysis
At 10% noise: which flags were *wrong* (flagged, but we never corrupted them)? These are often reviews that are
genuinely ambiguous or mislabeled in the original IMDb, which is an interesting finding in itself.

In [ ]:
seed = cfg["noise"]["seeds"][0]
flagged = pd.read_csv(f"{cfg['paths']['issues_dir']}/noise10_seed{seed}.csv")
false_flags = flagged[~flagged["is_noisy"]]
print(f"flagged: {len(flagged):,}   truly corrupted: {flagged.is_noisy.sum():,}   false flags: {len(false_flags):,}")
false_flags[["id", "true_label", "label", "suggested_label", "label_quality", "text"]].head(10)